# HonestAlpha - entrainement MLP sur Google Colab

Objectif : utiliser le GPU gratuit de Colab pour produire les fichiers `results/pred_mlp*.csv`, `icrang_mlp*.csv`, `paired_tests.csv`, `bootstrap.csv`, puis les telecharger pour les remettre dans le projet local.

Avant de lancer : dans Colab, choisir **Runtime > Change runtime type > T4 GPU**.

## 1. Envoyer le projet

Depuis Windows, zippe le dossier du projet en excluant `.venv`, `__pycache__`, `.pytest_cache` et `catboost_info`, puis envoie le zip dans la cellule suivante.

Exemple PowerShell depuis `D:\HonestAlpha_1` :

```powershell
Compress-Archive -Path configs,data,src,tests,requirements.txt,pyproject.toml,README.md,detail.md,Makefile -DestinationPath HonestAlpha_1_colab.zip -Force
```

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
zip_name

In [ ]:
import os, shutil, zipfile
workdir = '/content/HonestAlpha_1'
if os.path.exists(workdir):
    shutil.rmtree(workdir)
os.makedirs(workdir, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(workdir)
os.chdir(workdir)
print(os.getcwd())
!find . -maxdepth 2 -type f | sort | head -80

## 2. Installer les dependances

Colab a souvent deja `torch` et `numpy`. On installe le reste, puis on verifie que le GPU est visible.

In [ ]:
!pip -q install -r requirements.txt

import torch, numpy, pandas
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)

## 3. Lancer le pipeline avec MLP

`run` refait les predictions de tous les modeles, y compris les 5 graines MLP. `report` regenere `results/table.csv`, les figures, et le bootstrap MLP vs Ridge.

In [ ]:
!python -m pytest -q

In [ ]:
!python -m src.run run
!python -m src.run report

## 4. Controler les resultats MLP

In [ ]:
import pandas as pd
table = pd.read_csv('results/table.csv')
display(table)

for path in ['results/ic_summary.csv', 'results/paired_tests.csv', 'results/bootstrap.csv', 'results/portfolio.csv']:
    print('\n' + path)
    try:
        display(pd.read_csv(path))
    except FileNotFoundError:
        print('absent')

## 5. Telecharger les resultats

Le zip telecharge contient `results/` et les figures. De retour en local, dezippe-le dans `D:\HonestAlpha_1` en remplacant les fichiers `results/` existants.

In [ ]:
!zip -qr /content/results_colab.zip results
from google.colab import files
files.download('/content/results_colab.zip')